# F-JEPA : prototype sur les données journalières Teeptrak

**Objectif.** Tester de bout en bout une approche JEPA (*Joint-Embedding Predictive Architecture*) pour anticiper les défauts machine. On le fait sur les seules données disponibles pour l'instant : les historiques Teeptrak journaliers des lignes **BIO 2** et **Démoulage**, du 01/09/2024 au 31/08/2026.

**Limite à garder en tête.** Ces données donnent un % du temps par catégorie et par jour. Elles ne contiennent ni capteurs ni arrêts horodatés. On ne prédit donc pas une panne à l'heure près. On prédit si **demain sera un jour à défaut machine élevé**. Le pipeline (nettoyage, fenêtres, JEPA, sonde) restera valable quand on aura des données plus fines.

**Plan**
1. Chargement et nettoyage (fusion des colonnes en double, familles de catégories)
2. Étiquette « jour à défaut machine élevé »
3. Fenêtres glissantes de 28 jours, découpage train / test dans le temps
4. Baselines : persistance, régression logistique, gradient boosting
5. JEPA : pré-entraînement auto-supervisé, sans étiquettes
6. Évaluation : sonde linéaire sur les embeddings JEPA et score de surprise
7. Conclusions

Les fichiers Excel d'origine sont seulement lus, jamais modifiés.

In [1]:
# Dépendances : pip install -r requirements.txt  (pandas, openpyxl, scikit-learn, torch, matplotlib)
import copy, os, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore", message="enable_nested_tensor")
pd.set_option("display.width", 160)

## 0. Paramètres

In [2]:
# Le notebook est dans F-JEPA/prototype/, les exports Teeptrak dans F-JEPA/input/
INPUT_DIR = Path(os.environ.get("FJEPA_INPUT", "../input"))
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


# Découpage temporel (pas de mélange passé/futur)
TRAIN_END = "2025-10-31"                   # train : 2024-09-01 -> 2025-10-31
TEST_START = "2025-11-01"                  # test  : 2025-11-01 -> 2026-08-31

WINDOW = 28                                # jours d'historique vus par les modèles
HORIZON = 1                                # on prédit le jour J+HORIZON
HIGH_FAULT_QUANTILE = 0.80                 # "jour à défaut élevé" = top 20 % (seuil calculé sur le train)

# Familles de catégories Teeptrak. Tout ce qui n'est listé nulle part = "defaut_machine".
# >>> À valider avec le terrain : c'est ce choix qui définit l'étiquette de panne. <<<
CLOSED = {"Fermeture", "Fin De Production"}
PERFORMANCE = {"Normal", "Sous Cadence", "Sur Cadence", "Ralentissement / Micro Arrêt"}
PLANNED = {
    "Pause", "Delta Pause", "Réunions", "Formation", "Salle Grise", "Métrologie", "Ccp",
    "Of Essai", "Maintenance Préventive", "Démarrage/Vidange", "Cycle Test",
    "Changement De Commande", "Delta Cgt Commande", "Changement De Lot", "Changement De Format",
    "Changement De Bobine Film", "Changement De Bobine Stickers", "Changement Markem",
    "Changement D'outil", "Attente Bobine", "Attente Fonds De Boites", "Attente Rouleur",
    "Passage Plateaux Vides", "Reglage Papier", "Tirage Papier", "Transpalette",
}
QUALITY = {
    "Produit Perdu", "Sous Poids", "Sur Poids", "Reprise Fromage", "Larges Et Mous",
    "Défaut Fromages Collés", "Collage Fromage", "Saumure",
}
UNKNOWN = {"Non Renseigné", "Unknown"}
FAMILIES = ["ferme", "performance", "planifie", "qualite", "inconnu", "defaut_machine"]


def family_of(category: str) -> str:
    for fam, members in [("ferme", CLOSED), ("performance", PERFORMANCE), ("planifie", PLANNED),
                         ("qualite", QUALITY), ("inconnu", UNKNOWN)]:
        if category in members:
            return fam
    return "defaut_machine"


# JEPA
D_MODEL = 64
N_LAYERS = 2
N_HEADS = 4
PRED_LAYERS = 1
EMA_MOMENTUM = (0.99, 0.999)               # montée linéaire pendant l'entraînement
EPOCHS = 100                               # ~1 à 2 min par seed sur CPU
BATCH_SIZE = 128
LR = 1e-3
MASK_BLOCKS = 2                            # nb de blocs de jours masqués par fenêtre
MASK_BLOCK_LEN = (3, 7)                    # longueur min/max d'un bloc
SEEDS = [0, 1, 2]

## 1. Chargement et nettoyage

Dans chaque export « histo », la ligne 1 contient les métadonnées et la ligne 2 l'en-tête. Ensuite vient une ligne par jour : chaque colonne est le % du temps passé dans une catégorie, et chaque ligne somme à 100.

Le nettoyage fusionne les colonnes en double : « Gerbeuse » / « Gerbeuse␣ », « Non Renseigné » ×2, « Maintenance Préventive » ×3.

In [3]:
def _clean_name(c: str) -> str:
    c = re.sub(r"\.\d+$", "", str(c))      # "Maintenance Préventive.1" -> doublon pandas
    return " ".join(c.split())             # espaces de fin / multiples


def find_histo_files():
    """Renvoie {nom_ligne: chemin} pour les exports 'histo' (1 ligne par jour)."""
    out = {}
    for p in sorted(INPUT_DIR.rglob("*.xlsx")):
        if "histo" not in p.name.lower():
            continue
        line = "BIO2" if "bio" in p.parent.name.lower() else "DEMOULAGE"
        out[line] = p
    if not out:
        raise FileNotFoundError(f"Aucun fichier *histo*.xlsx trouvé sous {INPUT_DIR}")
    return out


def load_line(path) -> pd.DataFrame:
    """DataFrame indexé par date, une colonne par catégorie (en % du temps), doublons fusionnés."""
    raw = pd.read_excel(path, header=1)
    dates = pd.to_datetime(raw.pop("Days"), format="%d/%m/%Y")
    raw = raw.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    raw.columns = [_clean_name(c) for c in raw.columns]
    df = raw.T.groupby(level=0).sum().T     # fusion des colonnes de même nom
    df.index = dates
    return df.sort_index()


def add_families(df: pd.DataFrame) -> pd.DataFrame:
    fam = pd.DataFrame(index=df.index)
    for f in FAMILIES:
        cols = [c for c in df.columns if family_of(c) == f]
        fam[f"fam_{f}"] = df[cols].sum(axis=1) if cols else 0.0
    open_time = (100.0 - fam["fam_ferme"]).clip(lower=0)
    fam["temps_ouvert"] = open_time
    fam["jour_ouvert"] = (df.get("Normal", 0) > 0) & (open_time > 5)
    # Part du temps ouvert perdue en défaut machine : c'est la quantité que l'on cherche à anticiper.
    fam["part_defaut"] = np.where(open_time > 5, fam["fam_defaut_machine"] / open_time.clip(lower=1e-6), 0.0)
    return fam


def load_all():
    """Renvoie {ligne: (categories_df, familles_df)}."""
    out = {}
    for line, path in find_histo_files().items():
        cats = load_line(path)
        out[line] = (cats, add_families(cats))
    return out


raw = load_all()
for line, (cats, fam) in raw.items():
    print(f"{line:10s} {cats.index.min():%d/%m/%Y} -> {cats.index.max():%d/%m/%Y}  {cats.shape[1]} catégories, "
          f"{fam['jour_ouvert'].sum()} jours ouverts / {len(fam)}")

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

### Familles de catégories

Ce regroupement définit ce qu'on appelle « défaut machine ». **Il faut le valider avec le terrain.** Toute catégorie qui n'est pas listée dans `CLOSED`, `PERFORMANCE`, `PLANNED`, `QUALITY` ou `UNKNOWN` est comptée comme défaut machine.

In [ ]:
for line, (cats, _) in raw.items():
    print(f"--- {line} : catégories comptées comme défaut machine")
    print(", ".join(sorted(c for c in cats.columns if family_of(c) == "defaut_machine")))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, (line, (_, fam)) in zip(axes, raw.items()):
    f = fam.copy()
    ax.plot(f.index, f["fam_performance"].rolling(7).mean(), label="performance (Normal ± cadence), moy. 7 j")
    ax.plot(f.index, f["fam_ferme"].rolling(7).mean(), label="fermé, moy. 7 j")
    ax2 = ax.twinx()
    pdf = f["part_defaut"].where(f["jour_ouvert"])
    ax2.plot(f.index, pdf.rolling(7, min_periods=1).mean(), color="C3", label="part défaut machine (temps ouvert), moy. 7 j")
    ax.set_title(line); ax.set_ylabel("% du jour"); ax2.set_ylabel("part du temps ouvert")
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

## 2. Features et étiquette

**Features par jour**, dans un espace commun aux deux lignes :
- log(1 + %) de chaque catégorie (0 si la catégorie n'existe pas sur la ligne) ;
- les familles, `part_defaut` et `jour_ouvert` ;
- le calendrier (jour de la semaine, jour de l'année).

La standardisation est calculée **sur le train uniquement**.

**Étiquette.** Un jour ouvert est un « jour à défaut élevé » si sa part de temps ouvert perdue en défaut machine dépasse le 80e percentile des jours ouverts du train. Le seuil est calculé séparément pour chaque ligne. Les jours fermés ne reçoivent pas d'étiquette.

In [ ]:
def build_features(data):
    """Matrice de features par jour, espace commun aux deux lignes.

    - log1p(% du temps) pour chaque catégorie (union des deux lignes, 0 si absente)
    - les 6 familles + part_defaut + jour_ouvert
    - calendrier : jour de semaine et jour de l'année (sin/cos)
    Standardisation calculée sur la période d'entraînement uniquement.
    """
    all_cats = sorted(set().union(*[set(c.columns) for c, _ in data.values()]))
    feats, meta = {}, {}
    for line, (cats, fam) in data.items():
        x = pd.DataFrame(0.0, index=cats.index, columns=all_cats)
        x[cats.columns] = cats
        x = np.sign(x) * np.log1p(x.abs())
        famf = fam[[f"fam_{f}" for f in FAMILIES]].apply(np.log1p)
        famf["part_defaut"] = fam["part_defaut"]
        famf["jour_ouvert"] = fam["jour_ouvert"].astype(float)
        dow = cats.index.dayofweek.values
        doy = cats.index.dayofyear.values
        cal = pd.DataFrame({
            "dow_sin": np.sin(2 * np.pi * dow / 7), "dow_cos": np.cos(2 * np.pi * dow / 7),
            "doy_sin": np.sin(2 * np.pi * doy / 365.25), "doy_cos": np.cos(2 * np.pi * doy / 365.25),
        }, index=cats.index)
        feats[line] = pd.concat([x, famf, cal], axis=1)
        meta[line] = fam
    train_rows = pd.concat([f.loc[:TRAIN_END] for f in feats.values()])
    mu, sd = train_rows.mean(), train_rows.std().replace(0, 1.0)
    for line in feats:
        feats[line] = ((feats[line] - mu) / sd).astype(np.float32)
    return feats, meta


def make_labels(meta):
    """Étiquette binaire par ligne : jour ouvert dont la part de défaut machine dépasse
    le quantile HIGH_FAULT_QUANTILE des jours ouverts du train (seuil propre à chaque ligne)."""
    labels, thresholds = {}, {}
    for line, fam in meta.items():
        tr = fam.loc[:TRAIN_END]
        thr = tr.loc[tr["jour_ouvert"], "part_defaut"].quantile(HIGH_FAULT_QUANTILE)
        y = (fam["part_defaut"] > thr).astype(float)
        y[~fam["jour_ouvert"]] = np.nan       # jour fermé : pas d'étiquette
        labels[line], thresholds[line] = y, thr
    return labels, thresholds


feats, meta = build_features(raw)
labels, thr = make_labels(meta)
LINES = sorted(feats)
COLS = list(feats[LINES[0]].columns)
for l in LINES:
    print(f"{l}: seuil 'défaut élevé' = {thr[l]:.1%} du temps ouvert, {int(np.nansum(labels[l]))} jours positifs")
print(f"{len(COLS)} features par jour")

## 3. Fenêtres glissantes et découpage temporel

Chaque exemple est une fenêtre de `WINDOW` = 28 jours, du jour J−27 au jour J. Le modèle doit prédire l'étiquette du jour J+1.

- **Train** : jour cible ≤ 31/10/2025, soit 14 mois couvrant une saison complète.
- **Test** : jour cible ≥ 01/11/2025, soit 10 mois.

Il n'y a aucun mélange passé/futur : une fenêtre de train ne voit jamais un jour du test.

In [ ]:
def make_windows(feats, labels):
    """Fenêtres glissantes [t-WINDOW+1 .. t] -> étiquette de t+HORIZON.

    Renvoie X (N, WINDOW, F), y (N,), line_id (N,), target_date (N,).
    Les fenêtres dont le jour cible est fermé sont conservées avec y=NaN (utiles au pré-entraînement)."""
    Xs, ys, ls, ds = [], [], [], []
    for li, line in enumerate(sorted(feats)):
        f = feats[line].values
        y = labels[line].values
        idx = feats[line].index
        for t in range(WINDOW - 1, len(f) - HORIZON):
            Xs.append(f[t - WINDOW + 1:t + 1])
            ys.append(y[t + HORIZON])
            ls.append(li)
            ds.append(idx[t + HORIZON])
    return (np.stack(Xs).astype(np.float32), np.array(ys, dtype=np.float32),
            np.array(ls), pd.DatetimeIndex(ds))


def split_masks(dates):
    """Masque train / test selon la date cible. Une fenêtre de train ne voit jamais le test."""
    train = dates <= pd.Timestamp(TRAIN_END)
    test = dates >= pd.Timestamp(TEST_START)
    return train, test


X, y, line, dates = make_windows(feats, labels)
tr, te = split_masks(dates)
lab = ~np.isnan(y)
trl, tel = tr & lab, te & lab
print(f"fenêtres : {len(X)} (train {tr.sum()}, test {te.sum()}) ; étiquetées train {trl.sum()} / test {tel.sum()}")
print(f"taux de positifs : train {y[trl].mean():.1%}, test {y[tel].mean():.1%}")

## 4. Baselines

Avant le JEPA, il faut savoir ce que des méthodes simples obtiennent. On utilise deux métriques sur le test :
- **ROC-AUC** : 0,5 correspond au hasard, 1 à un classement parfait.
- **PR-AUC** (*average precision*) : le hasard vaut le taux de positifs. Elle est plus parlante quand les positifs sont rares.

In [ ]:
def target_context(dates, line):
    """Infos connues à l'avance sur le jour cible : calendrier + ligne."""
    dow, doy = dates.dayofweek.values, dates.dayofyear.values
    return np.column_stack([np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7),
                            np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25),
                            line.astype(float)])


def handcrafted(X):
    """Features classiques : dernier jour complet + moyenne 7 j, moyenne et écart-type 28 j des familles."""
    fam_idx = [COLS.index(c) for c in COLS if c.startswith("fam_") or c in ("part_defaut", "jour_ouvert")]
    return np.concatenate([X[:, -1], X[:, -7:, fam_idx].mean(1), X[:, :, fam_idx].mean(1),
                           X[:, :, fam_idx].std(1)], axis=1)


rows = []
def score(name, p):
    for lname, sel in [("toutes", np.ones(tel.sum(), bool))] + [(n, line[tel] == i) for i, n in enumerate(LINES)]:
        yy = y[tel][sel]
        rows.append({"modele": name, "ligne": lname, "n": int(sel.sum()), "positifs": int(yy.sum()),
                     "roc_auc": roc_auc_score(yy, p[sel]), "pr_auc": average_precision_score(yy, p[sel])})


def logreg(C=0.05):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000, class_weight="balanced"))


ctx = target_context(dates, line)
pdef = COLS.index("part_defaut")
score("persistance (part défaut de la veille)", X[tel, -1, pdef])
score("moyenne 7 j de la part défaut", X[tel, -7:, pdef].mean(1))

H = np.concatenate([handcrafted(X), ctx], axis=1)
score("régression logistique (features manuelles)", logreg().fit(H[trl], y[trl]).predict_proba(H[tel])[:, 1])
gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0).fit(H[trl], y[trl])
p_gb = gb.predict_proba(H[tel])[:, 1]
score("gradient boosting (features manuelles)", p_gb)

pd.DataFrame(rows).query("ligne == 'toutes'").round(3)

## 5. Le JEPA

Chaque **jour est un token**. Pour chaque fenêtre de 28 jours, l'entraînement suit quatre étapes :
1. On **masque** 2 blocs de 3 à 7 jours consécutifs. Une fois sur deux, un des blocs est placé en fin de fenêtre, pour que le modèle apprenne aussi à « prévoir ».
2. L'**encodeur de contexte** (petit Transformer) ne voit que les jours visibles.
3. Le **prédicteur** doit retrouver l'*embedding* de chaque jour masqué.
4. Ces embeddings cibles viennent de l'**encodeur cible**, une moyenne mobile exponentielle (EMA) des poids de l'encodeur de contexte, sans gradient. C'est ce mécanisme, avec la normalisation des cibles, qui évite l'effondrement des représentations.

La perte se calcule **dans l'espace des embeddings**, pas sur les valeurs brutes : c'est ce qui distingue un JEPA d'un auto-encodeur masqué. **Aucune étiquette n'est utilisée** pendant le pré-entraînement.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_feat, n_lines, d=D_MODEL, layers=N_LAYERS, heads=N_HEADS):
        super().__init__()
        self.inp = nn.Linear(n_feat, d)
        self.pos = nn.Parameter(torch.randn(1, WINDOW, d) * 0.02)
        self.line = nn.Embedding(n_lines, d)
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, layers)
        self.norm = nn.LayerNorm(d)

    def forward(self, x, line, hide=None):
        """hide : (B, W) bool, True = jour masqué (invisible pour l'attention)."""
        h = self.inp(x) + self.pos + self.line(line)[:, None]
        h = self.tf(h, src_key_padding_mask=hide)
        return self.norm(h)


class Predictor(nn.Module):
    def __init__(self, d=D_MODEL, layers=PRED_LAYERS, heads=N_HEADS):
        super().__init__()
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, WINDOW, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, layers)
        self.out = nn.Linear(d, d)

    def forward(self, ctx, hide):
        h = torch.where(hide[..., None], self.mask_token.expand_as(ctx), ctx) + self.pos
        return self.out(self.tf(h))


def sample_mask(batch, gen):
    """Blocs contigus de jours masqués ; une fois sur deux, le dernier bloc colle à la fin
    de la fenêtre (le modèle apprend aussi à « prévoir » les jours suivants)."""
    W = WINDOW
    m = np.zeros((batch, W), dtype=bool)
    for b in range(batch):
        for k in range(MASK_BLOCKS):
            L = gen.integers(MASK_BLOCK_LEN[0], MASK_BLOCK_LEN[1] + 1)
            start = W - L if (k == 0 and gen.random() < 0.5) else gen.integers(0, W - L + 1)
            m[b, start:start + L] = True
    return torch.from_numpy(m)


class JEPA(nn.Module):
    def __init__(self, n_feat, n_lines):
        super().__init__()
        self.context = Encoder(n_feat, n_lines)
        self.target = copy.deepcopy(self.context)
        for p in self.target.parameters():
            p.requires_grad_(False)
        self.predictor = Predictor()

    def loss(self, x, line, hide):
        with torch.no_grad():
            tgt = self.target(x, line)
            tgt = F.layer_norm(tgt, tgt.shape[-1:])
        ctx = self.context(x, line, hide)
        pred = self.predictor(ctx, hide)
        return F.smooth_l1_loss(pred[hide], tgt[hide])

    @torch.no_grad()
    def ema_update(self, m):
        for pc, pt in zip(self.context.parameters(), self.target.parameters()):
            pt.mul_(m).add_(pc.detach(), alpha=1 - m)

    @torch.no_grad()
    def embed(self, x, line):
        """Représentation d'une fenêtre : dernier jour + moyenne de la fenêtre (encodeur cible)."""
        h = self.target(x, line)
        return torch.cat([h[:, -1], h.mean(1)], dim=1)

    @torch.no_grad()
    def surprise(self, x, line):
        """Score d'anomalie non supervisé : erreur de prédiction de l'embedding du dernier jour
        à partir des jours précédents."""
        hide = torch.zeros(x.shape[:2], dtype=torch.bool)
        hide[:, -1] = True
        tgt = F.layer_norm(self.target(x, line), (D_MODEL,))
        pred = self.predictor(self.context(x, line, hide), hide)
        return (pred[:, -1] - tgt[:, -1]).pow(2).mean(-1)

In [ ]:
def pretrain(X, line, seed, log_every=50):
    torch.manual_seed(seed)
    gen = np.random.default_rng(seed)
    model = JEPA(X.shape[-1], int(line.max()) + 1)
    params = list(model.context.parameters()) + list(model.predictor.parameters())
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.05)
    steps_per_epoch = int(np.ceil(len(X) / BATCH_SIZE))
    total = EPOCHS * steps_per_epoch
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=total, pct_start=0.1)
    Xt, Lt = torch.from_numpy(X), torch.from_numpy(line).long()
    m0, m1 = EMA_MOMENTUM
    step, history = 0, []
    for ep in range(EPOCHS):
        model.train()
        perm = torch.from_numpy(gen.permutation(len(X)))
        ep_loss = 0.0
        for i in range(0, len(X), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            hide = sample_mask(len(idx), gen)
            loss = model.loss(Xt[idx], Lt[idx], hide)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            sched.step()
            model.ema_update(m0 + (m1 - m0) * step / total)
            step += 1
            ep_loss += loss.item() * len(idx)
        history.append(ep_loss / len(X))
        if log_every and (ep + 1) % log_every == 0:
            print(f"    seed {seed} epoch {ep + 1:4d}  loss {history[-1]:.4f}")
    model.eval()
    return model, history

### Pré-entraînement

On pré-entraîne sur la période train uniquement, y compris les fenêtres dont le jour cible est fermé. L'opération est répétée avec 3 seeds, parce qu'avec si peu de données la variance d'un entraînement à l'autre compte.

In [ ]:
torch.set_num_threads(max(1, os.cpu_count() or 1))
models, histories = {}, {}
t0 = time.time()
for seed in SEEDS:
    models[seed], histories[seed] = pretrain(X[tr], line[tr], seed, log_every=25)
    torch.save(models[seed].state_dict(), RESULTS_DIR / f"jepa_seed{seed}.pt")
print(f"pré-entraînement : {time.time() - t0:.0f} s")

for s, h in histories.items():
    plt.plot(h, label=f"seed {s}")
plt.xlabel("epoch"); plt.ylabel("perte JEPA (smooth L1, espace latent)"); plt.legend(); plt.show()

## 6. Évaluation

Deux façons d'utiliser le JEPA :
- **Sonde linéaire.** On gèle l'encodeur et on entraîne une simple régression logistique sur l'embedding de la fenêtre, plus le calendrier du jour cible. C'est le protocole standard pour juger la qualité des représentations.
- **Score de surprise**, sans aucune étiquette : l'erreur du prédicteur quand on lui demande l'embedding du dernier jour à partir des 27 précédents. Un jour « inattendu » pourrait annoncer un problème.

In [ ]:
Xt, Lt = torch.from_numpy(X), torch.from_numpy(line).long()
probs = []
for seed, model in models.items():
    Z = model.embed(Xt, Lt).numpy()
    S = model.surprise(Xt, Lt).numpy()
    print(f"seed {seed}: écart-type moyen des embeddings = {Z.std(0).mean():.3f} (≈ 0 signifierait un effondrement)")
    Zc = np.concatenate([Z, ctx], axis=1)
    p = logreg().fit(Zc[trl], y[trl]).predict_proba(Zc[tel])[:, 1]
    probs.append(p)
    score(f"JEPA + sonde linéaire (seed {seed})", p)
    score(f"JEPA score de surprise, non supervisé (seed {seed})", S[tel])
p_jepa = np.mean(probs, axis=0)
score("JEPA + sonde linéaire (moyenne des 3 seeds)", p_jepa)

res = pd.DataFrame(rows)
res.to_csv(RESULTS_DIR / "metrics.csv", index=False)
allr = res[res.ligne == "toutes"].copy()
allr["modele"] = allr.modele.str.replace(r" \(seed \d\)", " (3 seeds)", regex=True)
summary = allr.groupby("modele", sort=False).agg(roc_auc=("roc_auc", "mean"), roc_auc_std=("roc_auc", "std"),
                                                 pr_auc=("pr_auc", "mean")).round(3).fillna("")
print(f"Test 01/11/2025 -> 31/08/2026, {tel.sum()} jours-ligne, {y[tel].mean():.1%} de positifs "
      f"(hasard : ROC-AUC 0.5, PR-AUC {y[tel].mean():.3f})")
summary

In [ ]:
# Détail par ligne
res[~res.modele.str.contains("seed \\d", regex=True)].pivot_table(index="modele", columns="ligne", values="roc_auc", sort=False).round(3)

In [ ]:
fig, axes = plt.subplots(len(LINES), 1, figsize=(12, 3.2 * len(LINES)))
for ax, (i, l) in zip(axes, enumerate(LINES)):
    sel = line[tel] == i
    d = dates[tel][sel]
    ax.plot(d, meta[l].loc[d, "part_defaut"].values, color="0.6", lw=1, label="part défaut machine réelle")
    ax.axhline(thr[l], color="0.3", ls=":", lw=1, label="seuil « défaut élevé »")
    ax2 = ax.twinx()
    ax2.plot(d, pd.Series(p_jepa[sel]).rolling(3, min_periods=1).mean(), color="C0", lw=1.2, label="proba JEPA (moy. 3 j)")
    ax2.plot(d, pd.Series(p_gb[sel]).rolling(3, min_periods=1).mean(), color="C1", lw=1.2, alpha=0.8, label="proba boosting (moy. 3 j)")
    ax.set_title(f"{l} : période de test, jours ouverts")
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "test_predictions.png", dpi=120); plt.show()

## 7. Conclusions (exécution du 07/10/2026)

**Résultat principal : sur ces données journalières, personne ne prédit bien le lendemain, et le JEPA pas mieux que les méthodes simples.**

| Modèle | ROC-AUC test | PR-AUC test |
|---|---|---|
| Hasard | 0,50 | 0,21 |
| Persistance (part défaut de la veille) | 0,57 | 0,26 |
| Gradient boosting (features manuelles) | 0,59 | 0,27 |
| JEPA + sonde linéaire (3 seeds) | 0,51 ± 0,05 | 0,23 |
| JEPA score de surprise (non supervisé) | 0,52 ± 0,02 | 0,23 |

**Lecture**
- Même les meilleures baselines ne dépassent pas ~0,58 de ROC-AUC. Un jour à défaut élevé est **très peu prévisible** à partir des % journaliers des jours précédents. Les défauts machine se jouent à l'échelle de la minute ou de l'heure, et cette information disparaît dans l'agrégat journalier.
- Le JEPA apprend bien des représentations : les embeddings ne s'effondrent pas (écart-type ≈ 0,8) et la perte reste stable. Mais il ne dispose que d'environ **800 fenêtres** d'entraînement, alors que ces modèles ont besoin de beaucoup de données non étiquetées. C'est attendu, et ça ne dit rien de ce que le JEPA donnera sur des données fines.
- La période de test est plus « défaillante » que le train (21 % de positifs contre 17,5 %), et la saisonnalité est forte. Avec une seule saison complète en train, la généralisation est fragile.

**Ce que ce prototype apporte quand même**
- Un pipeline complet et sans fuite temporelle : nettoyage, familles de catégories, étiquette, fenêtres, split dans le temps, baselines, JEPA, sonde, score de surprise. Il pourra être rebranché sur un journal d'arrêts ou des capteurs.
- Une liste de catégories « défaut machine » (cellule 7) **à faire valider par le terrain**.

**Prochaines étapes**
1. Obtenir le **journal des arrêts Teeptrak** (événements horodatés). On pourra alors travailler à la minute avec des étiquettes de panne précises. Côté code, il suffira de remplacer « jour » par « pas de temps » et d'élargir la fenêtre.
2. Si possible, ajouter des **signaux capteurs ou automate**. C'est là qu'un JEPA a un avantage réel sur des features manuelles.
3. Sur les données actuelles, les variantes les plus prometteuses sont : un horizon plus long (« la semaine prochaine sera-t-elle mauvaise ? »), une cible par sous-ensemble (Palonnier, Nacelles…), et du pré-entraînement sur les deux lignes avec davantage d'augmentations.
